In [1]:
import numpy as np
import pandas as pd
from ax.service.ax_client import AxClient, ObjectiveProperties
from ax.core.metric import Metric
from ax.core.outcome_constraint import OutcomeConstraint
from ax.core.types import ComparisonOp
from ax.modelbridge.generation_strategy import GenerationStrategy, GenerationStep
from ax.modelbridge.registry import Models
import matplotlib.pyplot as plt

In [2]:
def x_normalizer(X, var_array):
    def max_min_scaler(x, x_max, x_min):
        return (x-x_min)/(x_max-x_min)
    x_norm = []
    for x in (X):
           x_norm.append([max_min_scaler(x[i], 
                                         max(var_array[i]), 
                                         min(var_array[i])) for i in range(len(x))])
    return x_norm

def x_denormalizer(x_norm, var_array):
    def max_min_rescaler(x, x_max, x_min):
        return x*(x_max-x_min)+x_min
    x_original = []
    for x in (x_norm):
           x_original.append([max_min_rescaler(x[i], 
                                         max(var_array[i]), 
                                         min(var_array[i])) for i in range(len(x))])
    return x_original

def get_closest_value(given_value, array_list):
    absolute_difference_function = lambda list_value : abs(list_value - given_value)
    closest_value = min(array_list, key=absolute_difference_function)
    return closest_value

def get_closest_array(suggested_x, var_list):
    modified_array = []
    for x in suggested_x:
        modified_array.append([get_closest_value(x[i], var_list[i]) for i in range(len(x))])
    return np.array(modified_array)

In [3]:
# ----- Objective definitions -----
obj_1 = "coverage"        # maximize (0..1)
obj_2 = "uniformity"      # minimize (better uniformity = lower values)
obj_3 = "phase_purity"  # CONVERTED TO CONSTRAINT: must equal 1 (single phase)

# ----- Parameter ranges and discrete values -----
param_ranges = {
    "spin_speed_rpm": [500.0, 5000.0],
    "precursor_conc_mol_L": [0.1, 1.5],
    "anneal_temp_C": [70.0, 300.0]
}

spinspeed_vals = list(np.arange(500, 5000, 500, dtype=float))
conc_vals = [round(x, 2) for x in np.arange(0.1, 1.5, 0.1)]
anneal_vals = list(np.arange(70, 300, 10, dtype=float))
param_lists = [spinspeed_vals, conc_vals, anneal_vals]
param_names = ["spin_speed_rpm", "precursor_conc_mol_L", "anneal_temp_C"]

PARAMS = [
    {"name": "spin_speed_rpm_norm", "type": "range", "bounds": [0.0, 1.0], "value_type": "float"},
    {"name": "precursor_conc_mol_L_norm", "type": "range", "bounds": [0.0, 1.0], "value_type": "float"},
    {"name": "anneal_temp_C_norm", "type": "range", "bounds": [0.0, 1.0], "value_type": "float"},
]

# ----- Initial experimental data (Latin hypercube sampling) -----
X_train = pd.DataFrame([
    {"spin_speed_rpm": 1000, "precursor_conc_mol_L": 0.2, "anneal_temp_C": 280},
    {"spin_speed_rpm": 1500, "precursor_conc_mol_L": 0.9, "anneal_temp_C": 130},
    {"spin_speed_rpm": 4000, "precursor_conc_mol_L": 1.4, "anneal_temp_C": 200},
    {"spin_speed_rpm": 3000, "precursor_conc_mol_L": 0.7, "anneal_temp_C": 240},
    {"spin_speed_rpm": 2500, "precursor_conc_mol_L": 0.4, "anneal_temp_C": 90},
    {"spin_speed_rpm": 1500, "precursor_conc_mol_L": 1.1, "anneal_temp_C": 170},
    {"spin_speed_rpm": 1000, "precursor_conc_mol_L": 1.4, "anneal_temp_C": 220.0},
    {"spin_speed_rpm": 3500, "precursor_conc_mol_L": 0.2, "anneal_temp_C": 120.0},
    {"spin_speed_rpm": 4500, "precursor_conc_mol_L": 1.0, "anneal_temp_C": 260.0},
    {"spin_speed_rpm": 2000, "precursor_conc_mol_L": 0.6, "anneal_temp_C": 150.0},
    {"spin_speed_rpm": 2500, "precursor_conc_mol_L": 1.0, "anneal_temp_C": 70.0},
    {"spin_speed_rpm": 4500, "precursor_conc_mol_L": 0.7, "anneal_temp_C": 210.0},
])

# Normalize training data
X_train_array = X_train[param_names].values
X_train_normalized = x_normalizer(X_train_array, [list(param_ranges[p]) for p in param_names])
X_train_norm = pd.DataFrame(X_train_normalized, columns=[f"{p}_norm" for p in param_names])

# Experimental results (raw values)
y_train_raw = pd.DataFrame([
    {"coverage": 0.675, "uniformity": 7.749, "phase_purity": 2},
    {"coverage": 0.778, "uniformity": 5.184, "phase_purity": 2},
    {"coverage": 0.878, "uniformity": 4.159, "phase_purity": 1},
    {"coverage": 0.946, "uniformity": 6.671, "phase_purity": 2},
    {"coverage": 0.017, "uniformity": 9.808, "phase_purity": 2},  
    {"coverage": 0.969, "uniformity": 4.254, "phase_purity": 1},
    {"coverage": 0.973, "uniformity": 4.205, "phase_purity": 1},
    {"coverage": 0.986, "uniformity": 7.038, "phase_purity": 2},
    {"coverage": 0.925, "uniformity": 5.010, "phase_purity": 1},
    {"coverage": 0.955, "uniformity": 6.318, "phase_purity": 2},
    {"coverage": 0.911, "uniformity": 3.782, "phase_purity": 1},
    {"coverage": 0.889, "uniformity": 8.531, "phase_purity": 2},
    
])

y_train_raw['phase_purity_violation'] = np.where(
    y_train_raw['phase_purity'] == 1, 
    -0.1,  # Feasible: negative value (satisfies constraint <= 0)
    0.9    # Infeasible: positive value (violates constraint <= 0)
)

# Prepare data for BO (two objectives + one constraint)
y_train_bo = [
    {obj_1: y_train_raw.iloc[i][obj_1], 
     obj_2: y_train_raw.iloc[i][obj_2],
     'phase_purity_violation': y_train_raw.iloc[i]['phase_purity_violation']} 
    for i in range(len(y_train_raw))
]

gs = GenerationStrategy(steps=[
    GenerationStep(
        model=Models.BOTORCH_MODULAR,    # Pure BoTorch MOO step (skip Sobol)
        num_trials=-1,                   # unlimited
        max_parallelism=32,              # High parallelism for batch generation
    ),
])

ax_client = AxClient(generation_strategy=gs, verbose_logging=False)
# Define the constraint: phase_purity_violation <= 0 (i.e., phase_purity == 1)
phase_purity_violation_metric = Metric(name="phase_purity_violation", lower_is_better=True)

# Constraint: must be <= 0 to be feasible
phase_purity_constraint = OutcomeConstraint(
    metric=phase_purity_violation_metric,
    op=ComparisonOp.LEQ,  
    bound=0.0,
    relative=False,
)

ax_client.create_experiment(
    parameters=PARAMS,
    objectives={
        obj_1: ObjectiveProperties(minimize=False),          # maximize coverage
        obj_2: ObjectiveProperties(minimize=True),          # minimize log(uniformity)
    },
    outcome_constraints=["phase_purity_violation <= 0.0"],
)

# Seed the experiment with scaled objective data
for i in range(len(X_train_norm)):
    ax_client.attach_trial(X_train_norm.iloc[i].to_dict())
    ax_client.complete_trial(trial_index=i, raw_data=y_train_bo[i])


[INFO 10-13 17:10:00] ax.service.utils.instantiation: Due to non-specification, we will use the heuristic for selecting objective thresholds.
[INFO 10-13 17:10:00] ax.service.utils.instantiation: Created search space: SearchSpace(parameters=[RangeParameter(name='spin_speed_rpm_norm', parameter_type=FLOAT, range=[0.0, 1.0]), RangeParameter(name='precursor_conc_mol_L_norm', parameter_type=FLOAT, range=[0.0, 1.0]), RangeParameter(name='anneal_temp_C_norm', parameter_type=FLOAT, range=[0.0, 1.0])], parameter_constraints=[]).


[INFO 10-13 17:10:00] ax.core.experiment: Attached custom parameterizations [{'spin_speed_rpm_norm': 0.111111, 'precursor_conc_mol_L_norm': 0.071429, 'anneal_temp_C_norm': 0.913043}] as trial 0.
[INFO 10-13 17:10:00] ax.core.experiment: Attached custom parameterizations [{'spin_speed_rpm_norm': 0.222222, 'precursor_conc_mol_L_norm': 0.571429, 'anneal_temp_C_norm': 0.26087}] as trial 1.
[INFO 10-13 17:10:00] ax.core.experiment: Attached custom parameterizations [{'spin_speed_rpm_norm': 0.777778, 'precursor_conc_mol_L_norm': 0.928571, 'anneal_temp_C_norm': 0.565217}] as trial 2.
[INFO 10-13 17:10:00] ax.core.experiment: Attached custom parameterizations [{'spin_speed_rpm_norm': 0.555556, 'precursor_conc_mol_L_norm': 0.428571, 'anneal_temp_C_norm': 0.73913}] as trial 3.
[INFO 10-13 17:10:00] ax.core.experiment: Attached custom parameterizations [{'spin_speed_rpm_norm': 0.444444, 'precursor_conc_mol_L_norm': 0.214286, 'anneal_temp_C_norm': 0.086957}] as trial 4.
[INFO 10-13 17:10:00] ax.co

In [4]:
def generate_next_experiments(n_suggest=12):
    
    global ax_client  # Declare global at the beginning
    
    # Generate suggestions
    
    suggestions_normalized = []
    for i in range(n_suggest):
        try:
            params_norm, trial_index = ax_client.get_next_trial()
            arm_name = ax_client.experiment.trials[trial_index].arm.name
            suggestions_normalized.append({
                "trial_index": trial_index, 
                "arm_name": arm_name, 
                **params_norm
            })
            print(f"Generated suggestion {i+1}/{n_suggest}")
        except Exception as e:
            print(f"Error generating suggestion {i+1}: {e}")
            break

    if not suggestions_normalized:
        print("Failed to generate any suggestions!")
        return None, None
    
    suggestions_df = pd.DataFrame(suggestions_normalized)

    # Convert to original parameter space
    norm_cols = [f"{p}_norm" for p in param_names]
    X_suggestions_norm = suggestions_df[norm_cols].values
    X_suggestions_denorm = x_denormalizer(X_suggestions_norm, [list(param_ranges[p]) for p in param_names])
    
    suggestions_original = suggestions_df.copy()
    for i, param in enumerate(param_names):
        suggestions_original[param] = [x[i] for x in X_suggestions_denorm]

    # Snap to discrete values for practical experiments
    X_suggestions_discrete = get_closest_array(X_suggestions_denorm, param_lists)
    suggestions_discrete = pd.DataFrame(X_suggestions_discrete, columns=param_names)
    
    return suggestions_discrete, suggestions_original


next_experiments, next_experiments_continuous = generate_next_experiments(n_suggest=6)
    

Generated suggestion 1/6


/Users/shengfang/anaconda3/envs/BO/lib/python3.12/site-packages/ax/core/data.py:288: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return cls(df=pd.concat(dfs, axis=0, sort=True))


Generated suggestion 2/6


/Users/shengfang/anaconda3/envs/BO/lib/python3.12/site-packages/ax/core/data.py:288: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return cls(df=pd.concat(dfs, axis=0, sort=True))


Generated suggestion 3/6


...

Generated suggestion 4/6


/Users/shengfang/anaconda3/envs/BO/lib/python3.12/site-packages/ax/core/data.py:288: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return cls(df=pd.concat(dfs, axis=0, sort=True))


Generated suggestion 5/6


/Users/shengfang/anaconda3/envs/BO/lib/python3.12/site-packages/ax/core/data.py:288: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return cls(df=pd.concat(dfs, axis=0, sort=True))


Generated suggestion 6/6


In [5]:
print(next_experiments, next_experiments_continuous)

   spin_speed_rpm  precursor_conc_mol_L  anneal_temp_C
0          3500.0                   1.1           70.0
1          3500.0                   1.1          160.0
2          3000.0                   1.3          230.0
3          2500.0                   1.4          220.0
4          2500.0                   1.4           70.0
5          3000.0                   1.1          180.0    trial_index arm_name  spin_speed_rpm_norm  precursor_conc_mol_L_norm  \
0           12     12_0             0.708387                   0.717716   
1           13     13_0             0.611953                   0.719504   
2           14     14_0             0.557108                   0.843348   
3           15     15_0             0.489676                   0.920702   
4           16     16_0             0.437149                   0.935374   
5           17     17_0             0.577198                   0.715548   

   anneal_temp_C_norm  spin_speed_rpm  precursor_conc_mol_L  anneal_temp_C  
0           

In [6]:
next_experiments.to_csv("/Users/shengfang/Desktop/TRI/test_FAPbI3/next_experiments_round2.csv", index=False)


In [ ]:
ax_client.save_to_json_file("/Users/shengfang/Desktop/TRI/test_FAPbI3/BO_Round1_model.json")  # This is the important one

FileNotFoundError: [Errno 2] No such file or directory: ' /Users/shengfang/Desktop/TRI/test_FAPbI3/BO_Round1_model.json'